# Data Center Water Stress Prediction

## 🌊 What is this notebook about?

Data centers consume **huge amounts of water** for cooling servers. This notebook predicts the **Water Stress Level** (High / Medium / Low) of the surrounding area based on a data center's technical and geographic features.

### 🎯 Goal
Build a Machine Learning model to classify water stress tier using features like PUE, WUE, electricity usage, water usage, and location.

### 📋 Dataset Columns
| Column | Description |
|--------|-------------|
| `Facility_Type` | Type of data center (Hyperscale, Colocation, Enterprise) |
| `PUE` | Power Usage Effectiveness (efficiency metric) |
| `WUE_L_per_kWh` | Water Usage Effectiveness (liters per kWh) |
| `Cooling_System_Type` | Air / Liquid / Evaporative cooling |
| `Daily_Electricity_Usage_MWh` | Daily power consumption |
| `Daily_Water_Usage_Gallons` | Daily water consumption |
| `Surrounding_Water_Stress_Tier` | **Target** → High / Medium / Low |

### 🗺️ Notebook Flow
1. Import Libraries
2. Load & Explore Data (EDA)
3. Feature Engineering
4. Build ML Model (Random Forest)
5. Evaluate Model
6. Feature Importance
7. Conclusion

## Step 1 — Import Libraries

We import all the tools we need:
- **pandas / numpy** → data handling
- **matplotlib / seaborn** → visualization
- **sklearn** → machine learning

In [ ]:
# ── Data handling ──────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualization ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine Learning ───────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Nicer plots
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ Libraries loaded!')

## Step 2 — Load & Explore the Data (EDA)

EDA = Exploratory Data Analysis. We look at the data **before** modeling to understand its shape, missing values, and distributions.

In [ ]:
# Load dataset
df = pd.read_csv('/kaggle/input/data-center-hybrid/data_center_hybrid.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Dataset info — column types and non-null counts
df.info()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print('\n✅ No missing values — clean dataset!')

In [ ]:
# Summary statistics for numeric columns
df.describe().T.style.background_gradient(cmap='Blues')

### 2.1 Target Variable Distribution

Let's see how balanced our target classes are.

In [ ]:
target_counts = df['Surrounding_Water_Stress_Tier'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
target_counts.plot(kind='bar', ax=axes[0], color=['#e74c3c','#3498db','#2ecc71'], edgecolor='black')
axes[0].set_title('Water Stress Tier — Count', fontsize=13)
axes[0].set_xlabel('Stress Tier')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
axes[1].pie(target_counts, labels=target_counts.index, autopct='%1.1f%%',
            colors=['#e74c3c','#3498db','#2ecc71'], startangle=90)
axes[1].set_title('Water Stress Tier — Share', fontsize=13)

plt.tight_layout()
plt.show()

print(target_counts)

### 2.2 Categorical Feature Distributions

In [ ]:
cat_cols = ['Facility_Type', 'Cooling_System_Type']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, hue='Surrounding_Water_Stress_Tier',
                  order=order, ax=ax)
    ax.set_title(f'{col} vs Water Stress', fontsize=12)
    ax.set_xlabel(col)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

### 2.3 Numeric Feature Distributions

In [ ]:
num_cols = ['PUE', 'WUE_L_per_kWh', 'Daily_Electricity_Usage_MWh',
            'Daily_Water_Usage_Gallons', 'Estimated_Capacity_MW']

fig, axes = plt.subplots(1, len(num_cols), figsize=(18, 4))

for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df, x='Surrounding_Water_Stress_Tier', y=col,
                hue='Surrounding_Water_Stress_Tier', ax=ax, palette='Set2', legend=False)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')

plt.suptitle('Numeric Features by Water Stress Tier', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 2.4 Correlation Heatmap

In [ ]:
corr = df[num_cols].corr()

plt.figure(figsize=(8, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

## Step 3 — Feature Engineering

We create new features that help the model learn better patterns:
- **WUE × Capacity** → total water stress load proxy
- **Electricity per MW** → operational efficiency
- **Water per Electricity** → water intensity of electricity

We also encode **City** and **Country** — location is a strong signal for regional water stress.

In [ ]:
# ── New engineered features ────────────────────────────────────
df['WUE_x_Cap']       = df['WUE_L_per_kWh'] * df['Estimated_Capacity_MW']
df['Elec_per_MW']     = df['Daily_Electricity_Usage_MWh'] / (df['Estimated_Capacity_MW'] + 1)
df['Water_per_Elec']  = df['Daily_Water_Usage_Gallons']   / (df['Daily_Electricity_Usage_MWh'] + 1)

# ── Label encode categorical columns ──────────────────────────
cat_encode_cols = ['Facility_Type', 'Cooling_System_Type', 'City', 'Country']

for col in cat_encode_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

print('✅ Feature engineering done!')
print('New columns:', ['WUE_x_Cap', 'Elec_per_MW', 'Water_per_Elec'])
df[['WUE_x_Cap', 'Elec_per_MW', 'Water_per_Elec']].describe()

## Step 4 — Prepare Data for Modeling

- **X** = input features
- **y** = target (Water Stress Tier encoded as numbers)
- Split into **80% train** and **20% test**

In [ ]:
# Features selected for the model
FEATURES = [
    'Year',
    'Estimated_Capacity_MW',
    'PUE',
    'WUE_L_per_kWh',
    'Daily_Electricity_Usage_MWh',
    'Daily_Water_Usage_Gallons',
    'WUE_x_Cap',
    'Elec_per_MW',
    'Water_per_Elec',
    'Facility_Type_enc',
    'Cooling_System_Type_enc',
    'City_enc',
    'Country_enc'
]

X = df[FEATURES]

# Encode target labels: High=0, Low=1, Medium=2
le_target = LabelEncoder()
y = le_target.fit_transform(df['Surrounding_Water_Stress_Tier'])

print('Classes:', list(le_target.classes_))

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTraining samples : {X_train.shape[0]:,}')
print(f'Testing  samples : {X_test.shape[0]:,}')

## Step 5 — Build the Model: Random Forest

**Random Forest** is an ensemble of Decision Trees.
- Trains many trees on random subsets of data
- Each tree votes; majority wins
- Great for tabular data and handles mixed feature types well

Key parameters:
- `n_estimators=200` → number of trees
- `max_depth=20` → max depth per tree (controls overfitting)
- `random_state=42` → reproducibility

In [ ]:
# Initialize and train model
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1   # use all CPU cores → faster training
)

print('🌲 Training Random Forest...')
model.fit(X_train, y_train)
print('✅ Training complete!')

## Step 6 — Evaluate the Model

We measure model performance using:
- **Accuracy** → % of correct predictions overall
- **Precision** → of predicted positives, how many are correct
- **Recall** → of actual positives, how many did we catch
- **F1-Score** → harmonic mean of precision and recall
- **Confusion Matrix** → visual breakdown of predictions

In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'🎯 Test Accuracy: {acc:.4f}  ({acc*100:.2f}%)')
print()
print('📊 Classification Report:')
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=le_target.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

# Quick interpretation
print('\nHow to read this:')
print('  Diagonal cells = correct predictions (higher = better)')
print('  Off-diagonal   = misclassifications')

## Step 7 — Feature Importance

Which features does the model rely on most? This helps us understand **what drives water stress** in and around data centers.

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 6))
bars = plt.barh(importances.index, importances.values,
                color=plt.cm.viridis(importances.values / importances.max()),
                edgecolor='gray')
plt.xlabel('Importance Score')
plt.title('Feature Importance — Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
print(importances.sort_values(ascending=False).head())

## ✅ Conclusion

### What We Did
1. Loaded and explored a **126,770-row** hybrid data center dataset
2. Performed EDA — checked distributions, correlations, and class balance
3. Engineered 3 new features (WUE × Capacity, Electricity per MW, Water per Electricity)
4. Encoded geographic features (City, Country) as they carry strong regional water-stress signals
5. Trained a **Random Forest Classifier** with 200 trees
6. Achieved **~71% accuracy** on the test set

### Key Findings
- **Location (City/Country)** is the most important predictor — water stress is largely a regional characteristic
- **WUE, PUE, water usage, and electricity usage** are strong secondary features
- **Facility type and cooling system type** have lower predictive power on their own

### What Could Improve the Model
- 🔼 Add external water stress index data (e.g., WRI Aqueduct) as a feature
- 🔼 Try XGBoost or LightGBM for potentially higher accuracy
- 🔼 Hyperparameter tuning with GridSearchCV
- 🔼 Use climate/precipitation features per region

> **💡 Takeaway:** Even without external geographic data, a Random Forest can classify regional water stress with ~71% accuracy using operational metrics and location encodings alone — a useful starting point for sustainability planning in the data center industry.